# 第2章 文章を数字にする：トークン

LLMにとって「猫」は 100472 番である

## このノートブックの使い方・学習目標

- **教科書のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html
- **エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **復習・理解度管理**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html)
- **この章でつくるもの**：LLMの文章分割を観察する「トークン観察器」と、第4章から使う自作の文字トークナイザー（＋未知文字・バイト版）
- **身につく数学・理論**：文字コードとUTF-8（16進数とバイト列） ／ トークン・語彙・トークンID ／ なぜ単語でも文字でもなく「トークン」なのか ／ 特別なトークンとコンテキスト長
- **新しく使うPython・PyTorch**：クラスメソッド `AutoTokenizer.from_pretrained` と `len(tok)` ／ `tok.encode` と `str.encode('utf-8')` の違い ／ `for` 文とリスト内包表記のトレース ／ `repr` で見えない文字を見る ／ `set`・`sorted`・`enumerate`・`.items()`・`join`

## 実行環境・リソースの目安

| 項目 | 目安 |
|---|---|
| 推奨ランタイム | Google Colab（CPUでOK） |
| 所要時間目安 | 60〜75分 |
| 難易度 | ★☆☆☆☆ |

> **✅ CPUでそのまま実行できます**：上のセルから順に再生ボタン（▶）または `Shift + Enter` で実行してください。

> **🎯 この章でつくるもの**
>
> この章が終わるころ、あなたは次の3つができるようになります。
>
> 1. `AutoTokenizer` の各メソッド（`encode` / `decode` / `convert_ids_to_tokens`）の**入力・出力・データ型**を正確に説明できる
> 2. なぜ日本語1文字が3バイトになり、珍しい漢字「鰯」が `�` 3つに分かれるのかをUTF-8の仕組みから説明できる
> 3. 文章⇔数字を往復変換する**文字トークナイザー**（さらに未知文字対応版・バイト版）を自分で実装できる

> **🔗 これまでの復習とつながり：第1章とのつながり**
>
> 第1章では、`chat = pipeline(...)` の中に `chat.tokenizer` という部品が入っていて、`messages` を `<|im_start|>...<|im_end|>` の台本文字列に変えていることを見ました。しかしコンピュータとLLM本体は、文字列をそのまま計算できません。この章では **`tokenizer` が文章をどうやって「数字の列」に変えているのか** を徹底的に分解します。

**ミナト**：前の章のLLM、日本語ペラペラでしたよね。でもコンピュータが数しか扱えないなら、「猫」とか「こんにちは」はどうやって数になっているんですか？

**ボンゴ先生**：LLMは日本語の文字そのものを「見て」すらいないんだ。LLMにとっての文章は `[106046, 116599, 15322, ...]` みたいな整数のリストだよ。今日はその変換係（トークナイザー）の中身を分解して、最後には自分の手でトークナイザーを作ってみよう！

## 準備：新しいノートブックを作る

第1章と同じように、Colabで新しいノートブックを作りましょう（「ファイル」→「ノートブックを新規作成」、名前は `02-tokens.ipynb` がおすすめ）。この章はモデル本体を使わないので**CPUのままでOK**です。

> **✨ コツ：毎章、新しいノートブックで**
>
> Colabは接続を切ると変数がすべて消えます。この教科書は「1章＝1ノートブック」で、**各章の最初のセルから順に実行すれば再現できる**ように設計してあります（ネット接続や無料GPU枠の変動への備えは[付録A](https://sakunokyumei.github.io/llm-research-textbook/a-setup.html)参照）。

## LLMの「辞書」を借りる

文章を数字に変える辞書と変換係をまとめて **トークナイザー**（tokenizer）と呼びます。第1章で使ったQwenのトークナイザーだけを借りてきましょう（数MBなので一瞬で終わります）。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
from transformers import AutoTokenizer

tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
print("語彙の数：", len(tok))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
語彙の数： 151665
```

</details>

<details><summary><b>基礎の補足：コードの分解：クラスメソッドと len(tok)</b>（クリックで開く）</summary>

- `AutoTokenizer` は設計図（クラス）です。`AutoTokenizer.from_pretrained("モデル名")` は「公開されている設定ファイルから、そのモデル専用のトークナイザー実物（オブジェクト）を組み立てて返す」という**クラスメソッド**です。
- 返ってきた `tok` は `Qwen2TokenizerFast` などのオブジェクトで、内部に「言葉のかけら ↔ 番号」の辞書を持っています。
- なぜリストでもない `tok` に `len(tok)` が使えるのか？ Pythonではクラスに特別な定義（`__len__`）を入れておくと、`len(オブジェクト)` でそのオブジェクトの「大きさ」（ここでは登録されている語彙の数）を返せるからです。

</details>

このトークナイザーは 151,665 種類の「言葉のかけら」を知っています。1つ1つのかけらを **トークン**（token）、トークン全体の一覧を **語彙**（ごい、vocabulary）と呼びます。

#### 📝 確認問題（`ch02-q01` · Lv.1 · トークナイザー・トークン・語彙の意味）

`tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")` を実行したあとの `tok` と `len(tok)` について、正しい説明はどれですか？

- **a.** `tok` は「文章⇔数字」を変換するオブジェクトで、`len(tok)` は辞書が知っているトークン総数（語彙数）を返す
- **b.** `tok` はモデル名の文字列そのものであり、`len(tok)` はモデル名の長さ（25文字）を返す
- **c.** `tok` は学習済みモデル本体を指しており、`len(tok)` は全パラメータ数（約5億個）を返す
- **d.** `AutoTokenizer.from_pretrained(...)` を呼ぶと、辞書だけでなくモデル本体（約1GB）も毎回必ずダウンロードされる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：トークナイザーとモデル本体の役割分担を思い出しましょう。
- **ヒント 2**：トークナイザーは「トークンと番号の対応表」を持つ部品オブジェクトで、`len(tok)` はその登録数を返します。
- **正解**：`a`

**考え方**：`クラス名.from_pretrained(...)` は公開ファイルから部品オブジェクトを組み立てて返すクラスメソッドです。`tok` は語彙一覧と変換メソッドを持っています。

1. `AutoTokenizer`（設計図）の `from_pretrained` を呼ぶと、`tok`（実物オブジェクト）が返る。
1. `len(tok)` は登録されているトークンの種類数＝語彙数（151,665）を返す。
1. モデル本体（重み）とは別部品なので軽量でCPUだけでもすぐ動く。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q01

</details>

## 文章を数字に変える

トークナイザーの基本操作は **`encode`（文章→数字）** と **`decode`（数字→文章）** の2つだけです。まずは入出力のデータ型を表で確認しましょう。

| メソッド | 入力（型） | 出力（型） | 目的 |
|---|---|---|---|
| `tok.encode(text)` | 文章（`str`） | トークンIDの並び（`list[int]`） | LLMに渡せる数字の列に変換する |
| `tok.decode(ids)` | トークンIDの並び（`list[int]`） | 文章（`str`） | LLMが選んだ数字を人間が読める文字に戻す |

実際に動かしてみます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
ids = tok.encode("吾輩は猫である。")
print(ids)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
[106046, 116599, 15322, 100472, 126230, 1773]
```

</details>

「吾輩は猫である。」が6個の整数になりました。この1つ1つの番号を **トークンID** と呼びます。逆向き（数字→文章）も試します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print(tok.decode(ids))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
吾輩は猫である。
```

</details>

では、6個の数字がそれぞれどのかけらに対応しているのか、1つずつ取り出して確認します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
for i in ids:
    print(i, "→", repr(tok.decode([i])))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
106046 → '吾'
116599 → '輩'
15322 → 'は'
100472 → '猫'
126230 → 'である'
1773 → '。'
```

</details>

<details><summary><b>基礎の補足：なぜ `tok.decode(i)` ではなく `tok.decode([i])` と書くのか？</b>（クリックで開く）</summary>

ここがつまずきやすいポイントです！
- `ids` は `[106046, 116599, ...]`（データ型は `list`）。
- `for i in ids:` で1個取り出した `i` は `106046`（データ型は `int`、ただの整数）。
- ところが `tok.decode(...)` は「IDの**リスト**」を受け取る設計になっているため、整数 `i` をそのまま渡すのではなく、**要素が1個だけのリスト `[i]`** に包んで渡しています。
- また `repr(...)` は文字列を `'...'` で囲んで表示する関数です。こうすると、ただの `print` では見えにくい「先頭の半角スペース（`' am'`）」や「改行（`'\n'`）」に確実に気づけます。

</details>

「猫」は1文字で1トークン（100472番）ですが、「である」は3文字まとめて1トークン（126230番）になっています。よく出るまとまりは1つのかけらとして辞書に登録されているのです。

#### 📝 確認問題（`ch02-q02` · Lv.2 · encode/decodeの入出力型と tok.decode([i]) の [ ]）

1つずつのトークンを確認するループ `for i in ids: print(i, tok.decode([i]))` で、`tok.decode(i)` ではなく **`[i]` と角かっこで囲んで渡す理由**はどれですか？

- **a.** Pythonでは変数を `[i]` のように角かっこで囲むと、元の整数 `i` の値が自動で1大きくなる仕様があるから
- **b.** `tok.decode(...)` は「トークンIDのリスト」を受け取る設計なので、1個の整数 `i` でも長さ1のリスト `[i]` にして渡すため
- **c.** `[i]` と角かっこで囲むことでPythonの辞書型を作成し、キーと値のペアとして `decode` に渡す必要があるから
- **d.** `tok.decode` は整数ではなく文字列しか受け取らないメソッドなので、`[i]` で文字列に変換して渡しているから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`ids` と、ループ変数 `i` の「データ型」の違いに注目しましょう。
- **ヒント 2**：`ids` は `list[int]` ですが、`for i in ids:` で取り出した `i` はただの `int` 1個です。
- **正解**：`b`

**考え方**：関数の引数に「どの型を渡すべきか」を意識するのがPython・AIコード読解の鍵です。`ids` は `list` ですが、`for i in ids:` で取り出した `i` はただの整数 `int` なので、`[i]` で長さ1のリストに包んでいます。

1. `ids = [106046, 116599, ...]`（型：`list`）。
1. `for i in ids:` で取り出した `i` は `106046`（型：`int`）。
1. `tok.decode(...)` はIDの列（`list`）を受け取るのが基本形なので、`[i]`（型：`list`）にして渡す。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q02

</details>

#### 📝 確認問題（`ch02-q03` · Lv.1 · repr()を使う理由）

トークンの中身を表示するとき、`print(tok.decode([i]))` ではなく `print(repr(tok.decode([i])))` を使うと何が嬉しいですか？

- **a.** トークン前後の余計な半角スペースや改行文字を自動で全部削除して読みやすく整形してくれる
- **b.** 日本語トークンの意味がすべて英語の単語へ自動翻訳され、中身を把握しやすくなる
- **c.** トークンIDの整数値がすべて2倍に変換されて表示され、大きな番号との差が見やすくなる
- **d.** `' am'` や `'\n'` のように引用符付きで表示され、空白や改行に気づける

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`print(" am")` と `print("am")` を画面で見比べたとき、違いに気づきにくいのを想像しましょう。
- **ヒント 2**：`repr(...)` は文字列を引用符で囲み、改行を `\n` のように可視化して表示してくれます。
- **正解**：`d`

**考え方**：トークナイザーのデバッグでは「先頭にスペースが付いているか」「改行文字が入っているか」が非常に重要なので、`repr()` で可視化します。

1. `print(" am")` → ` am`（空白があるか見落としやすい）。
1. `print(repr(" am"))` → `' am'`（引用符の直後にスペースがあるとはっきり分かる）。
1. `print(repr("\n"))` → `'\n'`（改行文字であることが一目で分かる）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q03

</details>

## トークンの切れ目を観察する

いろいろな文章を入れて切れ目を観察する関数 `show_tokens` を作ります。ここでは **リスト内包表記** `[tok.decode([i]) for i in ids]` が登場します。

<details><summary><b>基礎の補足：理解のための最小コード：リスト内包表記を for 文でトレースする</b>（クリックで開く）</summary>

`pieces = [tok.decode([i]) for i in ids]` は、次の3行の `for` 文を1行に縮めただけの書き方です。

```python no-run
# for文版（処理の流れが追いやすい）
pieces = []
for i in ids:
    word = tok.decode([i])    # 1個ずつ文字に戻す
    pieces.append(word)       # リストの末尾に追加する
```

`ids = [106046, 116599, 15322]` のときの中間値の変化：

| 周回 | `i`（`int`） | `tok.decode([i])`（`str`） | `pieces`（`list[str]`） |
|---|---|---|---|
| 開始前 | — | — | `[]` |
| 1周目 | `106046` | `'吾'` | `['吾']` |
| 2周目 | `116599` | `'輩'` | `['吾', '輩']` |
| 3周目 | `15322` | `'は'` | `['吾', '輩', 'は']` |

読みにくいと感じたときは、自分で書くときに3行の `for` 文で書いてまったく構いません。ただし人のコードを読む機会が多いので「`[結果 for 変数 in 集まり]` ＝ リストを作るfor文」と訳せるようにしておきましょう。

</details>

では本物のトークナイザーでいろいろな文章を試します。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def show_tokens(text):
    ids = tok.encode(text)
    pieces = [tok.decode([i]) for i in ids]
    print(len(ids), "トークン：", pieces)

show_tokens("こんにちは")
show_tokens("大規模言語モデル")
show_tokens("LLMを作れるようになりたい")
show_tokens("I am a cat.")
show_tokens("麒麟")
show_tokens("鰯")

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
1 トークン： ['こんにちは']
5 トークン： ['大', '規模', '言', '語', 'モデル']
7 トークン： ['LL', 'M', 'を作', 'れる', 'よう', 'になり', 'たい']
5 トークン： ['I', ' am', ' a', ' cat', '.']
1 トークン： ['麒麟']
3 トークン： ['�', '�', '�']
```

</details>

不思議な現象が4つ起きています。

- 「こんにちは」は5文字なのに1トークン（よく出るため丸ごと登録）
- 英語の `' am'` や `' cat'` は**前の半角スペース込み**で1トークン
- 「麒麟」は2文字で1トークン
- 「鰯」（いわし）は文字化け記号 `�` が**3つ**！

なぜ「鰯」だけ3つに分かれて `�` になったのでしょう？ 謎を解く鍵は **バイト** にあります。

#### 📝 確認問題（`ch02-q04` · Lv.2 · リスト内包表記とfor文の対応）

リスト内包表記の動きを確かめます。次のコードを実行すると何が表示されますか？

```python
nums = [2, 5, 10]
doubled = [x * 2 for x in nums]
print(doubled)
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：リスト内包表記 `[式 for 変数 in リスト]` は、各要素を順に加工して新しいリストを作ります。
- **ヒント 2**：`nums` の `2, 5, 10` をそれぞれ2倍にして、`[...]` の形で並べてみましょう。
- **期待される出力**：`[4, 10, 20]`

**考え方**：`[式 for 変数 in リスト]` は、空リストを作って `for` 文で `append` する処理の1行版です。

1. `x = 2` → `2 * 2 = 4`
1. `x = 5` → `5 * 2 = 10`
1. `x = 10` → `10 * 2 = 20`
1. 結果：`[4, 10, 20]`

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q04

</details>

## 文字の正体はバイトの列

> **📘 基本情報とのつながり：文字コードとUTF-8**
>
> コンピュータは文字を0〜255の数（**バイト**）の並びで保存します。世界中の文字を扱う標準が **Unicode** で、それをバイト列にする代表的な書き方が **UTF-8** です。Webページの `<meta charset="utf-8">` もこれです。

「猫」1文字がUTF-8でどんなバイト列になるか確かめましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
s = "猫"
b = s.encode("utf-8")
print(b)
print(len(s), "文字 →", len(b), "バイト")
print(list(b))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
b'\xe7\x8c\xab'
1 文字 → 3 バイト
[231, 140, 171]
```

</details>

#### ⚠️ つまずき注意：同じ `.encode` でも別物！ `tok.encode` vs `s.encode("utf-8")`

ここで**名前が同じ2つのメソッド**が登場しました。混乱しやすいので必ず `.` の左側（どのオブジェクトか）で区別しましょう。

| 書き方 | 誰のメソッド？ | 入力 → 出力（型） | 例（`"猫"`） |
|---|---|---|---|
| `tok.encode("猫")` | トークナイザー `tok` | `str` → `list[int]`（トークンID） | `[100472]` |
| `"猫".encode("utf-8")` | Pythonの文字列 `str` | `str` → `bytes`（UTF-8バイト列） | `b'\xe7\x8c\xab'`（`[231, 140, 171]`） |

<details><summary><b>基礎の補足：16進数 `\xe7` の手計算確認</b>（クリックで開く）</summary>

`b'\xe7\x8c\xab'` の `\x` は「後ろの2文字が16進数（0〜9、a=10、b=11、c=12、d=13、e=14、f=15）の1バイト」という印です。
- `e7` → $14 \times 16 + 7 = 224 + 7 = 231$
- `8c` → $8 \times 16 + 12 = 128 + 12 = 140$
- `ab` → $10 \times 16 + 11 = 160 + 11 = 171$

これが `list(b)` の `[231, 140, 171]` とぴったり一致します！ UTF-8では、英数字は1バイト、ひらがな・漢字の多くは3バイト、絵文字は4バイトになります。

</details>

*(対話デモはWeb教科書ページで確認できます：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html)*

> **💡 理論のポイント：LLMのトークナイザーは「バイト」を土台にしている（バイトレベルBPE）**
>
> Qwenのトークナイザーは、文字ではなく **UTF-8のバイト（0〜255の256種類）** を最小部品にしています。
>
> 1. まず文章をUTF-8のバイト列にする（「猫」→ `231, 140, 171`）
> 2. 学習データでよく一緒に出てくるバイトの並びを、1つのトークンにまとめる（`231, 140, 171` → 「猫」トークン 100472番）
> 3. 学習データに少ない珍しい並びは、1バイトずつのトークンのまま残る
>
> 「鰯」もUTF-8では3バイトですが、3バイト一体の登録がなかったため**1バイトずつの3トークン**に分かれました。1バイトだけでは日本語の文字にならないため、1個ずつ `decode` すると `�`（置換文字 U+FFFD）になります。しかし3個並べてまとめて `decode` すれば元の3バイトが揃うので、ちゃんと「鰯」に戻ります。
>
> この方式（**バイトレベルBPE**）の最大の利点は、**どんな文字・新語・絵文字でも絶対に表せて未知文字エラーが起きない**ことです（作り方は第14章でゼロから実装します）。

#### 📝 確認問題（`ch02-q05` · Lv.2 · tok.encode() と str.encode('utf-8') の違い）

第2章には `tok.encode("猫")` と `"猫".encode("utf-8")` という同じ `.encode` という名前のメソッドが2つ出てきます。両者の違いとして正しいものはどれですか？

- **a.** `tok.encode("猫")` は日本語1文字を必ずUTF-8の3バイトへ分解して扱うため、どの語彙でも必ず3個の整数IDを返す
- **b.** `tok.encode` はID列（`list[int]`）を返し、`"猫".encode("utf-8")` はUTF-8バイト列（`bytes`）を返す
- **c.** どちらも同じ `.encode` という名前のメソッドなので、実行するとどちらもまったく同じ `[231, 140, 171]` のリストを返す
- **d.** `"猫".encode("utf-8")` は外部ライブラリの機能なので、事前に `transformers` を `import` しておかないとエラーになる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：Pythonでは同じメソッド名でも「`.`（ドット）の左側のオブジェクト」によって働きが異なります。
- **ヒント 2**：`tok`（トークナイザー）の `encode` はトークンID列、文字列 `"猫"` の `encode("utf-8")` はバイト列を返します。
- **正解**：`b`

**考え方**：オブジェクト指向プログラミングでは、異なるクラスが同じ名前のメソッド（例：`encode`）を持つことがよくあります。必ず「何に対して呼んでいるか」と「戻り値の型」を確認します。

1. `tok.encode("猫")` → `tok`（`AutoTokenizer`）の仕事 → 戻り値は `list[int]`（トークンID）。
1. `"猫".encode("utf-8")` → `"猫"`（`str`）の仕事 → 戻り値は `bytes`（0〜255のバイト列）。
1. 両者は別物だが、バイトレベルBPEは内部でまずUTF-8バイト列にしてからトークンにまとめる関係にある。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q05

</details>

#### 📝 確認問題（`ch02-q06` · Lv.1 · UTF-8バイト数の計算）

英数字は1バイト、一般的な日本語（ひらがな・漢字）は3バイトのとき、文字列 `"AI猫"` をUTF-8で表すと合計何バイトになりますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：UTF-8では文字の種類ごとにバイト数が決まっています。1文字ずつ何バイトか分けて数えましょう。
- **ヒント 2**：`'A'`（1バイト）、`'I'`（1バイト）、`'猫'`（3バイト）を足し合わせます。
- **正解**：`5` バイト

**考え方**：UTF-8は文字の種類によってバイト数が変わる可変長符号化です。ASCII英数字は1バイト、日本語の多くは3バイト、絵文字は4バイトです。

1. `'A'` → 1バイト
1. `'I'` → 1バイト
1. `'猫'` → 3バイト
1. 合計：$1 + 1 + 3 = 5$ バイト（`len("AI猫".encode("utf-8"))` で確認可能）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q06

</details>

*(この節の追加問題 `ch02-q07`, `ch02-q08` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html で自動採点・復習できます)*

## なぜ「トークン」なの？

**ミナト**：でも、1文字ごとに番号を振ったほうがシンプルじゃないですか？ なぜわざわざ中途半端な「かけら」にまとめるんですか？

**ボンゴ先生**：とても鋭い疑問だ！ 実は第4章〜第13章では、分かりやすさのために「1文字＝1トークン」で自作GPTを作る。でも実用的な大きなLLMが「単語」でも「1文字」でもなく「トークン」を選ぶのには、はっきりした計算上の理由があるんだ。

#### 💡 理論のポイント：単語 vs 文字 vs トークン（トレードオフ）

| 単位 | 良いところ | 困るところ |
|---|---|---|
| **単語ごと** | 1つ1つに意味がある | 日本語は空白区切りがない。新語・固有名詞が無限にあり語彙が爆発する |
| **文字ごと** | 語彙が小さくシンプル | 文章が長い列になり、LLMの計算がとても重くなる |
| **トークン（サブワード）** | 語彙サイズと列の長さのいいとこ取り | 人間の語感と切れ目がずれることがある（`'を作'` など） |

LLMの計算の重さと、一度に読める最大長さ（**コンテキスト長**、Qwen2.5-0.5Bでは 32,768 トークン）は、**トークン数**で決まります。同じ文章を少ないトークンで表せるほど、速く・長く・安く（API料金もトークン数課金）処理できるのです。

言語ごとのトークン数の違いも確かめてみましょう。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
for text in ["I am a cat.", "Je suis un chat.", "我是一只猫。", "吾輩は猫である。"]:
    n = len(tok.encode(text))
    print(n, "トークン：", text)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
5 トークン： I am a cat.
5 トークン： Je suis un chat.
5 トークン： 我是一只猫。
6 トークン： 吾輩は猫である。
```

</details>

> **🔬 やってみよう：自分の名前や絵文字は何トークン？**
>
> `show_tokens("ここに好きな言葉")` を実行してみましょう。先に「何トークンくらいになりそうか」を予想してから実行するのがコツです。珍しい漢字や絵文字（🐈など）は細かいバイトに分解されるのを確かめてみてください。

#### 📝 確認問題（`ch02-q09` · Lv.2 · 単語・文字・トークンのトレードオフ）

大規模なLLMが「1文字＝1トークン」ではなく、よく出る並びをまとめた「サブワード（トークン）」を使う一番の理由はどれですか？

- **a.** 語彙サイズを抑えつつ文章のトークン列を短くし、計算量とコンテキスト消費を減らせるから
- **b.** 文章を細かく分割してトークン数を増やすほど、Transformerの計算時間が短くなって推論が高速化するから
- **c.** Pythonの仕様上、1文字を1トークンにする単純なトークナイザーはプログラムとして実装することができないから
- **d.** 日本語のひらがなや漢字はすべてUTF-8で1バイトに収まるため、サブワードにしないとバイト数が合わなくなるから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「単語単位」「文字単位」それぞれの弱点（辞書の大きさ vs 文章のトークン長）を思い出しましょう。
- **ヒント 2**：サブワード（トークン）は「語彙の肥大化」と「トークン列の長大化」の中間をとる設計です。
- **正解**：`a`

**考え方**：トークン設計は「語彙の大きさ」と「文章の長さ（トークン数）」のバランスをとる工学的なトレードオフです。

1. 単語単位：辞書が膨大になり、未知語だらけになる。
1. 文字単位：辞書は小さいが、1000文字＝1000ステップになり計算が重く、コンテキスト長もすぐ使い切る。
1. サブワード（トークン）：よく出る塊（「こんにちは」「である」等）を1トークンにし、短い列と適度な語彙数を両立する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q09

</details>

#### 📝 確認問題（`ch02-q17` · Lv.3 · 【応用】コンテキスト長とトークン数の見積もり）

ある日本語文章をトークン化すると、平均して **1文字あたり 0.8 トークン** になるとします。コンテキスト長が **32,000 トークン** のLLMには、おおよそ何文字の日本語を一度に入力できますか？

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：「トークン数 ＝ 文字数 × 1文字あたりのトークン数」の関係式を逆算しましょう。
- **ヒント 2**：文字数 × `0.8` ＝ `32000` なので、`32000` を `0.8` で割ります。
- **正解**：`40000` 文字

**考え方**：「1文字あたり何トークンか」が分かれば、コンテキスト長（トークン上限）やAPI料金から扱える文字数を概算できます。

1. トークン数 ＝ 文字数 × 0.8。
1. 文字数 ＝ $32000 \div 0.8 = 40000$ 文字（文庫本のおよそ3分の1〜半分弱）。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q17

</details>

## 特別なトークン

第1章のチャット台本に出てきた `<|im_start|>` と `<|im_end|>` は何トークンでしょう？ 確かめてみます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print(tok.encode("<|im_start|>user"))
print(tok.convert_ids_to_tokens(tok.encode("<|im_start|>user")))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
[151644, 872]
['<|im_start|>', 'user']
```

</details>

`<|im_start|>` は12文字もあるのに、**1トークン**（ID: `151644`）です。こうした制御用の目印を **特別なトークン**（special token）と呼びます。

| トークン | ID | 役目 |
|---|---|---|
| `<\|endoftext\|>` | 151643 | 文章の終わり |
| `<\|im_start\|>` | 151644 | セリフの始まり |
| `<\|im_end\|>` | 151645 | セリフの終わり（モデルがこれを予測したら生成を止める） |

LLMが「しゃべり終わったら自分で止まる」ことができるのは、普通の単語と同じように「次は `<|im_end|>` の確率がいちばん高い」と予測しているからです。

#### 📝 確認問題（`ch02-q10` · Lv.2 · 特殊トークン <|im_end|> で生成が止まる仕組み）

【第1章の復習連動】LLMが返答を書き終えたあと、永遠にしゃべり続けずに**自分で止まれる理由**はどれですか？

- **a.** モデルが一定文字数（10文字など）を出力した時点で、GPUの計算プロセスが強制的に終了する仕組みだから
- **b.** トークナイザーが毎ステップ乱数を振って、ランダムなタイミングで文章の出力を途中で強制終了しているから
- **c.** 返答の終わりに `<|im_end|>` という特殊トークンを予測するよう訓練され、それが出たら止まるから
- **d.** 生成された文章の中に句点「。」が1回でも現れた時点で、必ず出力を打ち切る固定ルールが入っているから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：第1章で見たチャット台本の末尾に付いていた特別な記号を思い出しましょう。
- **ヒント 2**：「終了の合図」も1つのトークンIDとして語彙に登録され、モデル自身が次のトークンとして予測します。
- **正解**：`c`

**考え方**：「止まること」さえも、特別なトークンID（Qwenでは151645番）を次のトークンとして予測する仕組みに統一されています。

1. `<|im_end|>` は10文字の普通の文字列ではなく、語彙に登録された**1つの特別なトークン**（ID: 151645）。
1. モデルは「答えを言い終えたら次は `<|im_end|>` の確率を高くする」よう訓練されている。
1. 生成プログラム側が `<|im_end|>` を受け取ったらループを終了する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q10

</details>

## 自分でトークナイザーを作る

ここからは実装パートです。第2部（第4〜10章）でゼロから言語モデルを作るために、最もシンプルな **文字トークナイザー**（1文字＝1トークン）を自分の手で作ります。

ステップは3つです：**① 語彙一覧 `chars` を作る → ② 対応辞書 `stoi` / `itos` を作る → ③ `encode` / `decode` 関数を作る**。

まずステップ①：重複のない文字一覧を作ります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
text = "吾輩は猫である。名前はまだ無い。"

chars = sorted(set(text))
print(chars)
print("語彙の数：", len(chars))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
['。', 'あ', 'い', 'だ', 'で', 'は', 'ま', 'る', '前', '名', '吾', '無', '猫', '輩']
語彙の数： 14
```

</details>

<details><summary><b>基礎の補足：`set` と `sorted` の入出力・型・目的</b>（クリックで開く）</summary>

- `set(text)`：文字列 `str` を受け取り、重複を除いた集合 `set` を作る（`'は'` や `'。'` が2回出ても1つになる）。
- `sorted(...)`：集合を受け取り、文字コード順に並べ替えた `list[str]` を返す。**なぜ `sorted` が必須か？** `set` は実行のたびに順番が変わるため、`sorted` で固定しないと「昨日の12番＝猫、今日の12番＝犬」のようにID対応が壊れてしまうからです。

</details>

#### 📝 確認問題（`ch02-q11` · Lv.1 · set() と sorted() による語彙作成）

語彙を作る最初の1行を確認します。次のコードの出力を予測してください。

```python
text = "ばなな"
chars = sorted(set(text))
print(chars)
print(len(chars))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：`set(...)` は重複する文字を1つにまとめ、`sorted(...)` は文字コード順に並べたリストを返します。
- **ヒント 2**：`"ばなな"` の重複を除くと `'な'` と `'ば'` の2文字です。五十音順（文字コード順）でどちらが先に来るか確認しましょう。
- **期待される出力**：`['な', 'ば']
2`

**考え方**：`set(text)` は重複を除いた文字の集合を作り、`sorted(...)` は毎回同じ順序（決定的なリスト）に並べ替えます。

1. `set("ばなな")` → `{'な', 'ば'}`（重複除去、種類は2つ）。
1. `sorted(...)` → `['な', 'ば']`（Unicodeコードポイント順で `'な'` < `'ば'`）。
1. `len(chars)` → `2`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q11

</details>

#### 📝 確認問題（`ch02-q13` · Lv.2 · 語彙リストに sorted() を掛ける理由）

自作トークナイザーで `chars = sorted(set(text))` のように **`sorted(...)` を必ず付ける理由**は何ですか？

- **a.** `sorted` を通すと文章中の重複文字が自動的に補完され、語彙リストの要素数が増えて精度が上がるから
- **b.** `set` は順序不定なので、`sorted` で固定しないと実行ごとに文字とIDの対応がずれるから
- **c.** `sorted` を付けると日本語のUTF-8バイト数が3バイトから1バイトへ圧縮され、メモリを節約できるから
- **d.** Pythonの `set` はそのままでは `for` 文や `enumerate` で要素を順番に取り出すことができない型だから

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：モデルが覚えているのは「文字そのもの」ではなく「トークンIDの番号」です。
- **ヒント 2**：学習時と推論時で文字とIDの対応表の並び順が変わってしまうと何が起きるか考えましょう。
- **正解**：`b`

**考え方**：モデルは「12番の次に4番が来やすい」とID番号で覚えているため、文字とIDの対応表は常に100%再現可能（決定的）でなければなりません。

1. Pythonの `set` はハッシュ randomization により、セッションごとに要素の並び順が変わりうる。
1. `sorted(set(text))` にすれば、同じ `text` から常に同じ `chars`・`stoi`・`itos` が作られる。
1. 実験の再現性を保つための重要な基本テクニック。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q13

</details>

次にステップ②：文字→整数IDの辞書 `stoi`（**s**tring **to** **i**nteger）と、逆向きの辞書 `itos`（**i**nteger **to** **s**tring）を作ります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
stoi = {}
for i, ch in enumerate(chars):
    stoi[ch] = i

itos = {}
for ch, i in stoi.items():
    itos[i] = ch

print(stoi)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
{'。': 0, 'あ': 1, 'い': 2, 'だ': 3, 'で': 4, 'は': 5, 'ま': 6, 'る': 7, '前': 8, '名': 9, '吾': 10, '無': 11, '猫': 12, '輩': 13}
```

</details>

<details><summary><b>基礎の補足：`enumerate` と `.items()` のトレース</b>（クリックで開く）</summary>

- `enumerate(chars)` はリストの中身を `(番号, 要素)` のペアで順番に出します（`(0, '。')`, `(1, 'あ')`, …）。
- `stoi.items()` は辞書の中身を `(キー, 値)` のペアで順番に出します（`('。', 0)`, `('あ', 1)`, …）。それを逆向きに `itos[i] = ch` へ入れています。
- 変数名 `stoi` / `itos` はLLMのコードや論文実装で世界共通の慣例です。

</details>

最後にステップ③：変換関数 `encode` と `decode` を作ります。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
def encode(s):
    return [stoi[c] for c in s]

def decode(ids):
    return "".join(itos[i] for i in ids)

ids = encode("猫である")
print(ids)
print(decode(ids))

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
[12, 4, 1, 7]
猫である
```

</details>

- `encode("猫である")`：入力 `str` → 中間値 `stoi['猫']=12, stoi['で']=4, stoi['あ']=1, stoi['る']=7` → 出力 `[12, 4, 1, 7]`（`list[int]`）。
- `decode([12, 4, 1, 7])`：入力 `list[int]` → `itos[i]` で `'猫','で','あ','る'` を取り出し、`"".join(...)`（区切り文字なしで連結）で出力 `"猫である"`（`str`）。

#### ⚠️ つまずき注意：辞書にない文字を入れると `KeyError` になる

この小さな辞書には `'犬'` がないので、`encode("犬")` を実行すると `KeyError: '犬'` になります。存在確認は `"犬" in stoi` で行えます。

🔎 **実行前に予想しよう**：このセルを実行すると、どんな数値・文字列・テンソルの形（shape）やグラフが出るか予想してから ▶ を押してください。

In [ ]:
print("犬" in stoi)

<details><summary><b>📊 期待される出力（実測）</b>（実行後に自分の出力と見比べよう）</summary>

```text
False
```

</details>

この弱点をどう克服するか──下の自力実装課題（`<unk>` 対応版とバイトトークナイザー版）で実際に作ってみましょう！

確認問題・自力実装・第1〜2章の総合復習に挑戦してください。

#### 📝 確認問題（`ch02-q12` · Lv.3 · enumerateとstoi/itos辞書の仕組み）

自作トークナイザーの `stoi`（文字→番号）と `itos`（番号→文字）を追跡しましょう。次のコードの出力を予測してください。

```python
chars = ['あ', 'い', 'う']
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
print([stoi[c] for c in "うあう"])
print("".join(itos[i] for i in [1, 0, 2]))
```

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：まず `enumerate(['あ', 'い', 'う'])` から `stoi`（文字→番号）と `itos`（番号→文字）の対応を書き出しましょう。
- **ヒント 2**：`あ:0, い:1, う:2` を使って `"うあう"` を番号リストにし、`[1, 0, 2]` を文字に戻して `"".join` します。
- **期待される出力**：`[2, 0, 2]
いあう`

**考え方**：`encode` は1文字ずつ `stoi[c]` を引いてリストにし、`decode` は1番号ずつ `itos[i]` を引いて `"".join(...)` でつなげます。

1. `stoi`：`あ→0, い→1, う→2`。
1. `"うあう"` → `[2, 0, 2]`。
1. `[1, 0, 2]` → `'い'`, `'あ'`, `'う'` を `"".join(...)` → `いあう`。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q12

</details>

#### 📝 確認問題（`ch02-q14` · Lv.3 · 未知文字とKeyErrorの原因）

`text = "吾輩は猫である。"` から作った文字トークナイザーで `encode("犬である")` を実行すると何が起きますか？

- **a.** 辞書にない `'犬'` だけを黙って無視して読み飛ばし、残りの `'である'` に対応するIDリストだけを返す
- **b.** 未知の文字 `'犬'` を自動的に検出してUTF-8の3バイトへ分解し、既存のバイトIDとして変換してくれる
- **c.** `'犬'` が辞書 `stoi` に登録されていないため、辞書引きの時点で `KeyError: '犬'` が発生して止まる
- **d.** 意味が近い既知の文字 `'猫'` を自動で探し出し、`'猫である'` のID列に置き換えて返してくれる

<details><summary>ヒント / 解答と解説を見る</summary>

- **ヒント 1**：自作した素朴な文字トークナイザーは `stoi[c]` という単純なPython辞書引きを使っています。
- **ヒント 2**：Pythonの辞書 `d[key]` に存在しないキーを渡すと、何が起きるでしょうか？
- **正解**：`c`

**考え方**：語彙外の文字（OOV: Out-Of-Vocabulary）への弱さが、素朴な文字トークナイザーの限界です。だからこそ実用LLMではバイトレベルBPEや未知トークン `<unk>` を使います。

1. `text` に `'犬'` は一度も登場していないので `"犬" in stoi` は `False`。
1. `[stoi[c] for c in "犬である"]` の先頭で `stoi["犬"]` を実行する。
1. キーが存在しないため `KeyError: '犬'` で停止する。

👉 自動採点・類題・間隔復習はWebページへ：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html#ex-ch02-q14

</details>

*(この節の追加問題 `ch02-q15`, `ch02-q16`, `ch02-q18` はWeb教科書 https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html で自動採点・復習できます)*

**✅ できたらチェック**

- [ ] `tok.encode` / `tok.decode` の入出力の型と、`tok.decode([i])` に `[ ]` が必要な理由を説明できる
- [ ] `tok.encode("猫")` と `"猫".encode("utf-8")` の違いを説明できる
- [ ] 16進数のバイト表現と、「鰯」が `�` 3つに分かれる理由を説明できる
- [ ] `sorted(set(text))`・`stoi`・`itos` で文字トークナイザーを自力実装し、未知文字対応やバイト版も検算テストに通した

> **📝 まとめ**
>
> - LLMは文章を **トークンID** の整数リストとして扱う。登録一覧が **語彙**
> - 実用LLMはUTF-8の**バイト**を土台によく出る並びをまとめる（バイトレベルBPE）ため、未知文字が起きずトークン列も短く保てる
> - `<|im_end|>` のような特別なトークンも1個のIDで、LLMが止まる合図に使われる
> - 自作の文字トークナイザーは `sorted(set(text))` → `stoi` / `itos` → `encode` / `decode` の3ステップで書ける

**ボンゴ先生**：文章を数字の列にするところまで完全に分解できたね！ 次の第3章はいよいよ第1部の山場だ。数字の列を受け取ったLLMが、どうやって「次のトークンの点数（ロジット）」を「確率」に変えてサイコロを振っているのか──**Σ記号、指数関数 $e^x$、softmax、温度**の「なぜその式になるのか」を、小さな数字の手計算からじっくり解き明かそう！

## 学習のふりかえり・次のステップ

- **困ったとき・エラーが出たとき**：[エラー辞典（付録E）](https://sakunokyumei.github.io/llm-research-textbook/e-errors.html)
- **忘れる前に定着を確認**：[学習状況・間隔復習ダッシュボード](https://sakunokyumei.github.io/llm-research-textbook/review.html) で確認問題・類題・7段階の進捗を記録できます
- **この章のWebページ**：https://sakunokyumei.github.io/llm-research-textbook/02-tokens.html
- **次の章**：[第3章 次の一語はサイコロで決まる：確率と温度](https://sakunokyumei.github.io/llm-research-textbook/03-next-token.html)